# Reçete Okuyucu — Colab'da L4 testi

Bu notebook, reçete okuyucuyu sunucudaki kurulumla aynı GPU'da (NVIDIA L4, 24 GB) dener:

1. vLLM ile `Qwen/Qwen3-VL-8B-Instruct` modelini başlatır (Docker yerine doğrudan süreç olarak).
2. İki örnek reçeteyle (spec Ek A) entegrasyon testlerini çalıştırır.
3. Okuyucu servisini başlatır; kendi görselinizi yükleyip JSON yanıtı görebilirsiniz.

**Gereksinim:** *Çalışma zamanı → Çalışma zamanı türünü değiştir → **L4 GPU*** (Colab Pro). T4'te 8B model sığmaz ve bf16 desteklenmez.

**Süre:** Kurulum ~5–10 dk, model indirme (~17 GB) ve açılış ~5–10 dk.

**KVKK:** Colab'a yüklenen her görsel Google'ın bulutuna gider. Burada yalnızca kişisel bilgileri karartılmış örnekler kullanın; gerçek doğrulanmış reçetelerle toplu değerlendirmeyi kendi sunucunuzda yapın.

**Özel repo ise:** Colab'ın sol menüsündeki 🔑 *Secrets* bölümüne `GITHUB_TOKEN` adıyla, bu repoyu okuma yetkisi olan bir GitHub token'ı ekleyin ve notebook erişimini açın.

In [ ]:
# 1) GPU kontrolü
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(out.stdout or out.stderr)
if "L4" not in out.stdout:
    print("UYARI: L4 GPU seçili değil. Sonuçlar ve süreler sunucuyu yansıtmayabilir; T4'te model sığmaz.")

## 2) Repoyu klonla

In [ ]:
import base64, os, shutil, subprocess

REPO = "FatihBrck/recete"
BRANCH = "feat/recete-okuyucu"  # Pull Request birleştirildikten sonra "main" yapın
DEST = "/content/recete"

token = None
try:
    from google.colab import userdata
    token = (userdata.get("GITHUB_TOKEN") or "").strip() or None
except Exception as exc:
    print(f"GITHUB_TOKEN okunamadı ({type(exc).__name__}). Repo özelse Secrets ayarını kontrol edin.")

if os.path.isdir(DEST) and not os.path.isdir(os.path.join(DEST, ".git")):
    shutil.rmtree(DEST)  # yarım kalmış klon

if not os.path.isdir(DEST):
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    if token:
        # Token komut satırına veya URL'ye yazılmaz: hata mesajlarında ve .git/config'te görünmez.
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1",
                   GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {basic}")
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, f"https://github.com/{REPO}.git", DEST],
                       env=env, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.replace(token, "***") if token else r.stderr)
        raise RuntimeError(
            "Klonlama başarısız. 'Authentication failed' veya 'not found' görüyorsanız: token'ın bu repoya "
            "erişimi ve 'Contents: Read-only' izni olduğunu, Secret adının GITHUB_TOKEN olduğunu ve "
            "notebook erişiminin açık olduğunu kontrol edin.")

%cd /content/recete
!git log --oneline -1

## 3) Kurulum

vLLM, Colab'ın kendi PyTorch kurulumuyla çakışmasın diye ayrı bir sanal ortama kurulur (sürüm `docker-compose.yml` ile aynı). Okuyucu servisi Colab'ın Python'una kurulur.

In [ ]:
!pip install -q uv
!uv venv -q /content/vllm-env --python 3.12
!uv pip install -q --python /content/vllm-env/bin/python "vllm==0.34.0"
!pip install -q -e ".[dev]"

## 4) vLLM'i başlat

Ayarlar `docker-compose.yml` ile aynıdır. `USE_FP8 = True` ile FP8 açılır (daha hızlı; doğruluğu 8c ve 9. hücrelerle karşılaştırın). İlk çalıştırmada model indirilir; hücre sunucu hazır olana kadar bekler (en fazla 30 dk). Hata olursa logun son kısmı yazdırılır.

In [ ]:
import subprocess, time, urllib.request

def wait_for(url, proc, timeout_s, log_path):
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            print(open(log_path).read()[-4000:])
            raise RuntimeError(f"süreç beklenmedik şekilde kapandı (kod {proc.returncode})")
        try:
            urllib.request.urlopen(url, timeout=5)
            print(f"hazır: {url} ({time.time() - t0:.0f} sn)")
            return
        except Exception:
            time.sleep(10)
    print(open(log_path).read()[-4000:])
    raise TimeoutError(url)

USE_FP8 = False   # True: L4'te ~1,6-2x daha hızlı üretim; doğruluğu karşılaştırarak deneyin

VLLM_LOG = "/content/vllm.log"
vllm_proc = None

def start_vllm(fp8):
    # vLLM'i (yeniden) başlatır. Model ilk seferden sonra diskten yüklenir (~2-3 dk).
    global vllm_proc
    if vllm_proc is not None and vllm_proc.poll() is None:
        vllm_proc.terminate(); vllm_proc.wait()
    cmd = [
        "/content/vllm-env/bin/vllm", "serve", "Qwen/Qwen3-VL-8B-Instruct",
        "--dtype", "bfloat16",
        "--max-model-len", "8192",
        "--gpu-memory-utilization", "0.92",
        "--max-num-seqs", "4",
        "--limit-mm-per-prompt", '{"image":1,"video":0}',
        "--port", "8000",
    ] + (["--quantization", "fp8"] if fp8 else [])
    vllm_proc = subprocess.Popen(cmd, stdout=open(VLLM_LOG, "w"), stderr=subprocess.STDOUT)
    wait_for("http://localhost:8000/v1/models", vllm_proc, 1800, VLLM_LOG)
    print("FP8:", fp8)

start_vllm(USE_FP8)

## 5) Örnek reçetelerle entegrasyon testi

`tests/integration/test_golden.py`: el yazısı (A.1) ve basılı form (A.2) örnekleri gerçek modelle okunur; değerler, güven puanının hesaplanabildiği ve sağ göz SPH güveninin > 0.5 olduğu kontrol edilir.

In [ ]:
!READER_VLLM_URL=http://localhost:8000/v1 python -m pytest -m integration -v

## 6) Okuyucu servisini başlat

Colab'ın kendi sunucusu 8080 portunu kullandığı için API **8081**'de açılır. `OUTPUT_FORMAT = "compact"` daha kısa model çıktısı kullanır (daha hızlı; `handwritten` belge bazında olur). `READER_VLLM_BASE_URL`, `config/models.yaml` içindeki Docker adresini (`http://vllm:8000/v1`) Colab'daki yerel adresle değiştirir.

In [ ]:
import os, subprocess

OUTPUT_FORMAT = "full"   # "compact": daha kısa model çıktısı (~%35-40), handwritten belge bazında

API_LOG = "/content/reader.log"
api_proc = None

def start_api(output_format):
    # Okuyucu servisini (yeniden) başlatır.
    global api_proc
    if api_proc is not None and api_proc.poll() is None:
        api_proc.terminate(); api_proc.wait()
    env = dict(os.environ, READER_VLLM_BASE_URL="http://localhost:8000/v1",
               READER_OUTPUT_FORMAT=output_format)
    api_proc = subprocess.Popen(
        ["python", "-m", "uvicorn", "--factory", "reader.api:create_app", "--host", "127.0.0.1", "--port", "8081"],
        cwd="/content/recete", env=env, stdout=open(API_LOG, "w"), stderr=subprocess.STDOUT,
    )
    wait_for("http://127.0.0.1:8081/health", api_proc, 120, API_LOG)
    print("Çıktı biçimi:", output_format)

start_api(OUTPUT_FORMAT)

## 7) Örnek bir reçeteyi API üzerinden oku

In [ ]:
import json, httpx

def read_image(name, data):
    r = httpx.post("http://127.0.0.1:8081/v1/read", files={"file": (name, data)}, timeout=120)
    print(name, "->", r.status_code)
    print(json.dumps(r.json(), ensure_ascii=False, indent=2))

path = "tests/fixtures/golden/a1_handwritten.webp"
read_image(path, open(path, "rb").read())

## 8) Kendi görselinizi deneyin

JPEG, PNG, WEBP veya HEIC, en fazla 15 MB. **Kişisel bilgileri karartılmış** görseller kullanın (KVKK).

In [ ]:
from google.colab import files

uploaded = files.upload()
for name, data in uploaded.items():
    read_image(name, data)

## 8b) Tanılama: modelin ham çıktısı

Bir reçete beklenmedik sonuç verirse (ör. `no_values_read`), aynı görseli servisi atlayarak doğrudan modele gönderir ve iki aşamanın **ham JSON** çıktısını gösterir. Servis bu çıktıyı KVKK nedeniyle loglamaz; burada yalnızca ekranda görürsünüz. 8. hücrede yüklenen görseller kullanılır.

In [ ]:
import json
from reader.config import Settings, load_model_config, load_prompt_set
from reader.llm_client import OpenAICompatClient
from reader.preprocess import load_image, fit_max_pixels, to_jpeg_b64
from reader.schemas import Classification, LensExtraction, GlassesExtraction
from reader.compact import LensExtractionCompact, GlassesExtractionCompact

cfg = load_model_config(Settings(vllm_base_url="http://localhost:8000/v1", output_format=OUTPUT_FORMAT))
ps = load_prompt_set(cfg)
client = OpenAICompatClient.from_config(cfg)

print("Yüklenen dosyalar:", list(uploaded))
name = list(uploaded)[-1]          # incelemek istediğiniz dosyanın adını yazın
img = fit_max_pixels(load_image(uploaded[name]), cfg.max_image_pixels)
b64 = to_jpeg_b64(img)
print("Modele giden boyut:", img.size)

cls = await client.complete(system=ps.classify.system, user=ps.classify.user, image_jpeg_b64=b64,
                            schema_name="Classification", json_schema=Classification.model_json_schema())
doc_type = json.loads(cls.text)["doc_type"]
print("Sınıflandırma:", cls.text)

if doc_type != "other":
    tpl = {"lens_printed": ps.lens_printed, "lens_handwritten": ps.lens_handwritten}.get(doc_type, ps.glasses)
    if cfg.output_format == "compact":
        schema = GlassesExtractionCompact if doc_type == "glasses" else LensExtractionCompact
    else:
        schema = GlassesExtraction if doc_type == "glasses" else LensExtraction
    ext = await client.complete(system=tpl.system, user=tpl.user, image_jpeg_b64=b64,
                                schema_name=schema.__name__, json_schema=schema.model_json_schema())
    print(f"Okuma ({tpl.version}):")
    print(json.dumps(json.loads(ext.text), ensure_ascii=False, indent=2))
await client.aclose()

## 8c) Hız karşılaştırması

Aynı görselleri farklı ayarlarla okutup süreleri ve değerleri yan yana gösterir. Her ayar için birkaç tekrar yapılır; ilk istek ısınma olduğu için sayılmaz.

- Çıktı biçimi (`full` / `compact`) yalnızca okuyucu servisini yeniden başlatır (birkaç sn).
- FP8'i açıp kapatmak vLLM'i yeniden başlatır (~2-3 dk). Varsayılan olarak yalnızca çıktı biçimi karşılaştırılır; FP8'i de denemek için `TRY_FP8 = True` yapın.

Değerler ayarlar arasında farklılaşıyorsa hız kazancı doğruluk pahasına geliyor demektir; karar vermeden önce doğrulanmış reçetelerle değerlendirme (9. hücre) yapın.

In [ ]:
import statistics, httpx

TRY_FP8 = False
IMAGES = ["tests/fixtures/golden/a1_handwritten.webp", "tests/fixtures/golden/a2_printed.webp"]
REPEATS = 3

def values(resp):
    out = {}
    for eye in ("right", "left"):
        e = resp.get(eye) or {}
        for f in ("sph", "cyl", "axis", "add", "bc", "dia"):
            v = (e.get(f) or {}).get("value")
            if v is not None:
                out[f"{eye}.{f}"] = v
    return out

def bench(label):
    rows = []
    for path in IMAGES:
        data = open(path, "rb").read()
        httpx.post("http://127.0.0.1:8081/v1/read", files={"file": (path, data)}, timeout=120)  # ısınma
        times, extract, last = [], [], None
        for _ in range(REPEATS):
            r = httpx.post("http://127.0.0.1:8081/v1/read", files={"file": (path, data)}, timeout=120).json()
            times.append(r["latency_ms"]); extract.append(r.get("stage_latency_ms", {}).get("extract", 0)); last = r
        rows.append((label, path.split("/")[-1], statistics.median(times), statistics.median(extract), values(last)))
    return rows

results = []
configs = [(False, "full"), (False, "compact")] + ([(True, "full"), (True, "compact")] if TRY_FP8 else [])
current_fp8 = USE_FP8
for fp8, fmt in configs:
    if fp8 != current_fp8:
        start_vllm(fp8); current_fp8 = fp8
    start_api(fmt)
    results += bench(f"FP8={fp8} / {fmt}")

print(f"{'ayar':<22}{'görsel':<22}{'toplam ms':>10}{'okuma ms':>10}")
for label, img, total, ext, _ in results:
    print(f"{label:<22}{img:<22}{total:>10.0f}{ext:>10.0f}")
print()
for label, img, _, _, vals in results:
    print(label, img, vals)

## 9) (İsteğe bağlı) Küçük bir değerlendirme

Manifest biçimi README'nin *Değerlendirme* bölümünde. Görselleri ve `manifest.jsonl` dosyasını `/content/data/` altına yükleyip aşağıdaki satırların başındaki `#` işaretini kaldırın. Asıl test setini (tür başına ~100 reçete) kendi sunucunuzda çalıştırın.

In [ ]:
# !python -m eval.run_eval run --manifest /content/data/manifest.jsonl --url http://127.0.0.1:8081 --out /content/preds.jsonl
# !python -m eval.run_eval report --manifest /content/data/manifest.jsonl --predictions /content/preds.jsonl --out /content/report
# print(open("/content/report/report.md").read())

## 10) Kapat

In [ ]:
if api_proc is not None:
    api_proc.terminate()
if vllm_proc is not None:
    vllm_proc.terminate()
print("Servisler durduruldu. Colab oturumu kapanınca indirilen model de silinir.")